
# Сверка фактического резерва `df_port` и расчетного резерва `df_out`

Ноутбук выполняет сверку для **первой отчетной даты**.

Логика:

- в `df_port` суммируется столбец `Резерв факт` по УНП;
- сумма делится на `1000`;
- в `df_out` расчетный резерв считается временно как:

  `OD × %рез / 100`

- расчетный резерв **не записывается в `df_out` отдельным столбцом**;
- расчетный резерв суммируется по УНП;
- выводятся только отклонения;
- отдельно отмечаются УНП, которые есть только в `df_out` или только в `df_port`;
- дополнительно выполняется общая сверка резервов по всему портфелю.

Перед запуском должны существовать DataFrame `df_port` и `df_out`.


In [ ]:

import re
import pandas as pd


## Настройки

In [ ]:

PORT_UNN_COL = "УНП"
PORT_RESERVE_COL = "Резерв факт"

OUT_UNN_COL = "УНП"


## Сверка резервов

In [ ]:

# =============================================================================
# 1. НОРМАЛИЗУЕМ УНП
# =============================================================================

df_port[PORT_UNN_COL] = (
    df_port[PORT_UNN_COL]
    .astype("string")
    .str.strip()
)

df_out[OUT_UNN_COL] = (
    df_out[OUT_UNN_COL]
    .astype("string")
    .str.strip()
)


# =============================================================================
# 2. НАХОДИМ ПЕРВУЮ ОТЧЕТНУЮ ДАТУ
# =============================================================================

report_dates = []

for col in df_out.columns:

    match = re.match(
        r"^задолженность_(\d{2}\.\d{2}\.\d{4})$",
        str(col)
    )

    if match:
        report_dates.append(
            match.group(1)
        )


if not report_dates:
    raise ValueError(
        "В df_out не найдены столбцы вида "
        "'задолженность_01.01.2026'"
    )


first_date = min(
    report_dates,
    key=lambda x: pd.to_datetime(
        x,
        format="%d.%m.%Y"
    )
)


OUT_OD_COL = (
    f"OD_{first_date}"
)

OUT_RATE_COL = (
    f"%рез_{first_date}"
)


if OUT_OD_COL not in df_out.columns:
    raise ValueError(
        f"В df_out отсутствует столбец {OUT_OD_COL}"
    )


if OUT_RATE_COL not in df_out.columns:
    raise ValueError(
        f"В df_out отсутствует столбец {OUT_RATE_COL}"
    )


print(
    "Первая отчетная дата:",
    first_date
)


# =============================================================================
# 3. ПРИВОДИМ ЧИСЛОВЫЕ ПОЛЯ К ЧИСЛАМ
# =============================================================================

port_reserve_numeric = (
    pd.to_numeric(
        df_port[PORT_RESERVE_COL],
        errors="coerce"
    )
    .fillna(0)
)


out_od_numeric = (
    pd.to_numeric(
        df_out[OUT_OD_COL],
        errors="coerce"
    )
    .fillna(0)
)


out_rate_numeric = (
    pd.to_numeric(
        df_out[OUT_RATE_COL],
        errors="coerce"
    )
    .fillna(0)
)


# =============================================================================
# 4. ВРЕМЕННО СЧИТАЕМ РАСЧЕТНЫЙ РЕЗЕРВ В df_out
# =============================================================================
#
# ВАЖНО:
# новый столбец в df_out НЕ создается.
#
# =============================================================================

out_calc_reserve = (
    out_od_numeric
    *
    out_rate_numeric
    /
    100
)


# =============================================================================
# 5. df_port:
#    СУММИРУЕМ РЕЗЕРВ ФАКТ ПО УНП И ДЕЛИМ НА 1000
# =============================================================================

port_reserve_sum = (
    pd.DataFrame({
        PORT_UNN_COL:
            df_port[PORT_UNN_COL],

        "резерв_факт_df_port":
            port_reserve_numeric,
    })
    .groupby(
        PORT_UNN_COL,
        as_index=False
    )["резерв_факт_df_port"]
    .sum()
)


port_reserve_sum[
    "резерв_факт_df_port"
] = (
    port_reserve_sum[
        "резерв_факт_df_port"
    ]
    / 1000
)


# =============================================================================
# 6. df_out:
#    СУММИРУЕМ РАСЧЕТНЫЙ РЕЗЕРВ ПО УНП
# =============================================================================

out_reserve_sum = (
    pd.DataFrame({
        OUT_UNN_COL:
            df_out[OUT_UNN_COL],

        "резерв_df_out":
            out_calc_reserve,
    })
    .groupby(
        OUT_UNN_COL,
        as_index=False
    )["резерв_df_out"]
    .sum()
)


# =============================================================================
# 7. ОБЪЕДИНЯЕМ ПО УНП
# =============================================================================
#
# outer нужен, чтобы сохранить:
#
# - УНП только из df_out
# - УНП только из df_port
#
# =============================================================================

reserve_check = (
    out_reserve_sum.merge(
        port_reserve_sum,
        on=OUT_UNN_COL,
        how="outer",
        indicator=True
    )
)


# =============================================================================
# 8. ПРОПУСКИ = 0
# =============================================================================

reserve_check[
    [
        "резерв_df_out",
        "резерв_факт_df_port"
    ]
] = (
    reserve_check[
        [
            "резерв_df_out",
            "резерв_факт_df_port"
        ]
    ]
    .fillna(0)
)


# =============================================================================
# 9. ОТКЛОНЕНИЕ
# =============================================================================

reserve_check[
    "отклонение"
] = (
    reserve_check[
        "резерв_df_out"
    ]
    -
    reserve_check[
        "резерв_факт_df_port"
    ]
)


# =============================================================================
# 10. СТАТУС
# =============================================================================

def get_reserve_status(row):

    if row["_merge"] == "left_only":
        return "УНП есть только в df_out"

    if row["_merge"] == "right_only":
        return "УНП есть только в df_port"

    if abs(
        row["отклонение"]
    ) > 0.01:
        return "Отклонение резерва"

    return "Совпадает"


reserve_check[
    "статус"
] = (
    reserve_check.apply(
        get_reserve_status,
        axis=1
    )
)


# =============================================================================
# 11. ОСТАВЛЯЕМ ТОЛЬКО ОТКЛОНЕНИЯ
# =============================================================================

reserve_deviations = (
    reserve_check.loc[
        reserve_check[
            "отклонение"
        ].abs() > 0.01
    ]
    .copy()
)


reserve_deviations.drop(
    columns="_merge",
    inplace=True
)


# =============================================================================
# 12. СОРТИРУЕМ ПО МОДУЛЮ ОТКЛОНЕНИЯ
# =============================================================================

reserve_deviations[
    "модуль_отклонения"
] = (
    reserve_deviations[
        "отклонение"
    ]
    .abs()
)


reserve_deviations = (
    reserve_deviations
    .sort_values(
        "модуль_отклонения",
        ascending=False
    )
    .drop(
        columns="модуль_отклонения"
    )
    .reset_index(
        drop=True
    )
)


# =============================================================================
# 13. ВЫВОД ОТКЛОНЕНИЙ ПО УНП
# =============================================================================

print(
    "\nОтклонения резервов по УНП:"
)

display(
    reserve_deviations
)


# =============================================================================
# 14. ОБЩИЙ РЕЗЕРВ ФАКТ df_port
# =============================================================================

total_port_reserve = (
    port_reserve_numeric.sum()
    / 1000
)


# =============================================================================
# 15. ОБЩИЙ РАСЧЕТНЫЙ РЕЗЕРВ df_out
# =============================================================================

total_out_reserve = (
    out_calc_reserve.sum()
)


# =============================================================================
# 16. ОБЩЕЕ ОТКЛОНЕНИЕ
# =============================================================================

total_reserve_deviation = (
    total_out_reserve
    -
    total_port_reserve
)


# =============================================================================
# 17. ОБЩАЯ ТАБЛИЦА КОНТРОЛЯ
# =============================================================================

total_reserve_check = pd.DataFrame({

    "дата": [
        first_date
    ],

    "резерв_факт_df_port": [
        total_port_reserve
    ],

    "резерв_df_out": [
        total_out_reserve
    ],

    "отклонение": [
        total_reserve_deviation
    ],

})


print(
    "\nОбщий контроль резервов:"
)

display(
    total_reserve_check
)


# =============================================================================
# 18. КРАТКИЙ ИТОГ
# =============================================================================

print(
    f"Первая отчетная дата: "
    f"{first_date}"
)

print(
    f"Резерв факт df_port / 1000: "
    f"{total_port_reserve:,.2f}"
)

print(
    f"Расчетный резерв df_out: "
    f"{total_out_reserve:,.2f}"
)

print(
    f"Общее отклонение: "
    f"{total_reserve_deviation:,.2f}"
)

print(
    f"Количество УНП с отклонениями: "
    f"{len(reserve_deviations)}"
)



## Результаты

После выполнения основной ячейки доступны:

- `reserve_deviations` — отклонения резервов по УНП;
- `total_reserve_check` — общий контроль резервов;
- `out_calc_reserve` — временная Series с расчетным резервом по каждой строке `df_out`;
- `first_date` — первая отчетная дата.

`df_out` при расчете не получает дополнительных столбцов.


In [ ]:
reserve_deviations.head()

In [ ]:
total_reserve_check